# Renal Evidence Studio

Notebook ini mendokumentasikan workflow proyek CKD screening: mengambil dataset, melatih model, membaca metrik, melihat feature importance, dan mencoba prediksi contoh.

Catatan: proyek ini hanya untuk edukasi dan screening berbasis machine learning, bukan diagnosis medis.

## 1. Setup

Jalankan cell ini dari root repository. Jika dependency belum terpasang, jalankan `pip install -r requirements.txt` terlebih dahulu.

In [4]:
from pathlib import Path
import json

import pandas as pd

ROOT = Path.cwd()
ROOT

WindowsPath('d:/Aku(rizal)/Telkom University Surabaya/Semester 6/SIstem Cerdas/Tubes')

## 2. Ambil dan Cache Dataset

Dataset utama berasal dari UCI Chronic Kidney Disease dataset id 336. Script fetch memiliki fallback offline agar notebook tetap bisa dijalankan ketika koneksi tidak tersedia.

In [5]:
from scripts.fetch_data import ensure_dataset, DATA_PATH, METADATA_PATH

dataset_path = ensure_dataset()
metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))

print(f"Dataset path: {dataset_path}")
print(f"Rows: {metadata['row_count']}")
print(f"Features: {metadata['feature_count']}")
print(metadata['citation'])

Dataset path: D:\Aku(rizal)\Telkom University Surabaya\Semester 6\SIstem Cerdas\Tubes\data\raw\ckd.csv
Rows: 400
Features: 24
Rubini, L., Soundarapandian, P., & Eswaran, P. (2015). Chronic Kidney Disease [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5G020


## 3. Eksplorasi Singkat Dataset

In [6]:
df = pd.read_csv(DATA_PATH)
df.head()

,age,blood_pressure,specific_gravity,albumin,sugar,red_blood_cells,pus_cell,pus_cell_clumps,bacteria,blood_glucose_random,...,packed_cell_volume,white_blood_cell_count,red_blood_cell_count,hypertension,diabetes_mellitus,coronary_artery_disease,appetite,pedal_edema,anemia,classification
0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,121.0,...,44.0,7800.0,5.2,yes,yes,no,good,no,no,ckd
1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,NaN,...,38.0,6000.0,NaN,no,no,no,good,no,no,ckd
2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,...,31.0,7500.0,NaN,no,yes,no,poor,no,yes,ckd
3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,...,32.0,6700.0,3.9,yes,no,no,poor,yes,yes,ckd
4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,...,35.0,7300.0,4.6,no,no,no,good,no,no,ckd


In [7]:
df['classification'].value_counts(dropna=False)

classification
ckd       250
notckd    150
Name: count, dtype: int64

In [8]:
missing_summary = (
    df.isna()
    .sum()
    .sort_values(ascending=False)
    .rename('missing_count')
    .to_frame()
)
missing_summary.head(12)

,missing_count
red_blood_cells,152
red_blood_cell_count,131
white_blood_cell_count,106
potassium,88
sodium,87
packed_cell_volume,71
pus_cell,65
hemoglobin,52
sugar,49
specific_gravity,47


## 4. Training dan Evaluasi

Training membandingkan Logistic Regression, Decision Tree, Random Forest, dan SVC. Model terbaik dipilih berdasarkan mean cross-validated F1 untuk label `ckd`.

In [9]:
from src.train import train_and_save

result = train_and_save(random_state=42)
result['selected_model']

d:\Aku(rizal)\Telkom University Surabaya\Semester 6\SIstem Cerdas\Tubes\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
d:\Aku(rizal)\Telkom University Surabaya\Semester 6\SIstem Cerdas\Tubes\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
d:\Aku(rizal)\Telkom University Surabaya\Semester 6\SIstem Cerdas\Tubes\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
d:\Aku(rizal)\Te

'random_forest'

In [11]:
metrics = result['metrics']
leaderboard = pd.DataFrame(
    [
        {
            'model': model_name,
            'mean_f1': values['mean_f1'],
            'std_f1': values['std_f1'],
        }
        for model_name, values in metrics['cv_scores'].items()
    ]
).sort_values('mean_f1', ascending=False)

leaderboard

,model,mean_f1,std_f1
2,random_forest,0.994999,0.006125
0,logistic_regression,0.994872,0.010256
3,svc_rbf,0.992405,0.006201
1,decision_tree,0.982269,0.017193


In [12]:
summary_metrics = {
    key: metrics[key]
    for key in ['selected_model', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'confusion_matrix']
}
summary_metrics

{'selected_model': 'random_forest',
 'accuracy': 0.9875,
 'precision': 0.9803921568627451,
 'recall': 1.0,
 'f1': 0.9900990099009901,
 'roc_auc': 1.0,
 'confusion_matrix': [[50, 0], [1, 29]]}

## 5. Feature Importance

Importance di bawah menjelaskan perilaku model pada data evaluasi. Ini bukan klaim kausal medis.

In [13]:
feature_importance_path = ROOT / 'app' / 'artifacts' / 'feature_importance.json'
feature_importance = json.loads(feature_importance_path.read_text(encoding='utf-8'))

pd.DataFrame(feature_importance['top_features'])

,feature,importance,std
0,specific_gravity,0.019436,0.012266
1,serum_creatinine,0.015648,0.010870
2,hemoglobin,0.008679,0.008058
3,hypertension,0.007728,0.007204
4,diabetes_mellitus,0.007728,0.007204
5,age,0.006795,0.004448
6,albumin,0.006757,0.007519
7,potassium,0.005824,0.004755
8,blood_urea,0.003883,0.004755
9,sodium,0.003883,0.004755


## 6. Contoh Prediksi Lokal

Cell ini memakai service predictor yang sama dengan API FastAPI.

In [14]:
from app.schemas import SAMPLE_PAYLOAD
from app.services.predictor import Predictor

predictor = Predictor()
prediction = predictor.predict(SAMPLE_PAYLOAD)
prediction

{'prediction': 'notckd',
 'probability': 0.9797223338506299,
 'top_features': [{'feature': 'specific_gravity',
   'importance': 0.019436146812853737},
  {'feature': 'serum_creatinine', 'importance': 0.01564840081952227},
  {'feature': 'hemoglobin', 'importance': 0.008679057511690647},
  {'feature': 'hypertension', 'importance': 0.007727785913942631},
  {'feature': 'diabetes_mellitus', 'importance': 0.007727785913942631}],
 'disclaimer': 'Educational screening only; not a medical diagnosis.'}

## 7. Menjalankan Web App

Dari terminal root repository:

```powershell
python -m uvicorn app.main:app --reload --port 8000
```

Lalu buka `http://127.0.0.1:8000` di browser.